

# MODULE 34 — SPEECH PROCESSING

# 34.9 Modern Speech Recognition Models 🎙️

Bro, now we're moving from ASR fundamentals to the actual deep-learning models used to convert speech into text!

In the previous module, we studied HMM, GMM, CTC, encoder-decoder architectures and Transformers. Now we'll see how these ideas are implemented in four important speech recognition models:

1. DeepSpeech
2. Wav2Vec 2.0
3. Whisper
4. Speech Transformer

We'll finish with a practical Whisper implementation using Python and PyTorch-compatible tools.

---

## 1. DeepSpeech

![DeepSpeech architecture illustration](https://images.openai.com/static-rsc-4/WQqCgLSGg6vT30Q7RDCvGBZfWtfJKdTVymleGBB9H8qKoTa3r2YWpn08DtpREsgqsnXNJZWNYl2PC421kb7tonGFVx_alAnu-3pOvxBm3LlRVkq_Cu2ObocGLf_9HVvE6aEvywgxS4WNRriciZZzO7VVlsxLii3Zys5eWs9EgMA?purpose=inline)

DeepSpeech

A deep-learning-based speech recognition architecture inspired by end-to-end speech recognition research.

DeepSpeech is associated with the development of end-to-end neural ASR. Mozilla popularized an open-source implementation based on Baidu's Deep Speech research.

### 1.1 Architecture

The original DeepSpeech architecture uses:

- Audio feature extraction.
- Multiple fully connected neural network layers.
- A bidirectional recurrent layer (in the original architecture).
- A final output layer.
- CTC for sequence training.

![DeepSpeech 2 implementation illustration](https://images.openai.com/static-rsc-4/3_DDl0tPcGTbkGaA_8tdG0RsVnn0GKCqJm8T-dBvoHAeCBZtO7sfiWuEyzhbMdgF7cet9GiX-qhfyBuwEGF2tr06pUtumV7KClbqkmWdKeZBOv-28oBZvDjv0cBZ3WpsJpfwJAR6fOwE4ZyOGviJTGMYPawJeFfOZFB33-VbG1s?purpose=inline)

Source: [GitHub — Deep Speech 2 PyTorch implementation](https://github.com/sooftware/deepspeech2)

**Conceptual flow:**

```text
Audio
  ↓
Spectral Features
  ↓
Neural Network Layers
  ↓
Bidirectional RNN
  ↓
Character Probabilities
  ↓
CTC Decoding
  ↓
Recognized Text
```

The network predicts character probabilities for successive audio frames. CTC helps convert these frame-level predictions into a text sequence without requiring manually aligned character boundaries.

### 1.2 Input Representation

The original DeepSpeech implementation uses audio-derived spectral features, commonly MFCC-style features.

These features are presented as a sequence:

$$
X=[x_1,x_2,\ldots,x_T]
$$

Each $x_t$ represents the acoustic features of one frame.

### 1.3 Training Approach

- Supervised learning using audio and corresponding transcripts.
- Neural network learns acoustic-to-character relationships.
- CTC loss handles unknown alignment between frames and output characters.
- A language model can optionally improve decoding.

### 1.4 Output Generation

The model predicts character probabilities, including a CTC blank symbol.

The decoder collapses repeated characters and removes blanks to produce the transcription.

### 1.5 Multilingual Support

DeepSpeech implementations were generally trained for particular languages and datasets. Multilingual recognition is not an automatic capability of the architecture; it requires suitable multilingual training data and model configuration.

### 1.6 Advantages and Limitations

| Advantages | Limitations |
|---|---|
| Relatively straightforward end-to-end design | Older architecture compared with modern large ASR models |
| CTC simplifies alignment | Performance depends heavily on training data |
| Character-based output | Language-specific training is often needed |
| Useful historically and educationally | Less capable of exploiting broad multilingual pretraining |

**Important:** DeepSpeech is valuable for understanding the evolution of neural ASR, but it is not the model I'd choose as the starting point for a new production project today.

---

## 2. Wav2Vec 2.0

![Wav2Vec 2.0 model illustration](https://images.openai.com/static-rsc-4/A1yEazoSotRtSRX9I75yo3zjyFHJsggEplnLOEKA6tBdh1uoDdv1QAM3G-OghxrD5_d3CMmVBY1lCKY7Y7XyyxGrNXbYAuLe9RIoyEFLKb_FTODTcOfRcNQiLy2CshZd62C_YEl4GHttCV2w5CJWyXKgFvc0KM3QHFpDv-MsSBI?purpose=inline)

Wav2Vec 2.0

Introduced by Facebook AI Research (now Meta AI), Wav2Vec 2.0 uses **self-supervised learning** to learn powerful speech representations from unlabelled audio.

This model is particularly important because it addresses a major challenge in speech recognition: obtaining large quantities of transcribed speech is expensive.

### 2.1 Architecture

Wav2Vec 2.0 has three main components:

1. **Feature Encoder:** Converts raw audio into latent speech representations.
2. **Context Network:** Uses Transformer layers to understand contextual relationships.
3. **Quantization Module:** Produces discrete representations used in its self-supervised training objective.

![Wav2Vec 2.0 architecture illustration](https://images.openai.com/static-rsc-4/w-zpkIHLysK6CLA3NmUxZSRLZ3P67n9tOVxOcCstL-p5OheURfHYLIloh0-0wuG28Mlh8utvlc_KSLZLZWHxhA6AXL4G0BSVG-GBSR3Kj0d_sL5lQXZ76OJZcZhubCKXU-EWCvpCxDjgOt6aQIQg5BZMLxCz91PHZeNBFuT684Y?purpose=inline)

Source: [John Snow Labs — Wav2Vec2 discussion](https://github.com/JohnSnowLabs/spark-nlp/discussions/12842)

**Conceptual flow:**

```text
Raw Audio
    ↓
CNN Feature Encoder
    ↓
Latent Representations
    ↓
Masking
    ↓
Transformer Context Network
    ↓
Contextual Representations
```

The quantization pathway supplies discrete target representations during pretraining. It is important to understand that quantization is part of the self-supervised learning mechanism, not simply a mandatory text-generation stage during transcription.

### 2.2 Input Representation

Unlike traditional MFCC-based systems, Wav2Vec 2.0 takes **raw audio waveforms** as input.

For example:

```python
audio_waveform = [
    0.12,
    -0.08,
    0.31,
    0.45,
    -0.22
]
```

The CNN feature encoder learns useful representations from the waveform.

### 2.3 Training Approach

Its most important idea is self-supervised pretraining.

The process:

1. Collect large quantities of unlabelled speech.
2. Convert raw audio into latent representations.
3. Mask portions of the latent sequence.
4. Use the Transformer context network to predict the correct quantized representations.
5. Fine-tune the pretrained model using labelled audio-transcript pairs for ASR.

The pretraining objective uses contrastive learning to distinguish the correct target representation from distractors.

### 2.4 Output Generation

A typical ASR fine-tuning setup adds a CTC output layer.

```text
Raw Audio
    ↓
CNN Encoder
    ↓
Transformer
    ↓
CTC Output Layer
    ↓
Character / Token Probabilities
    ↓
CTC Decoder
    ↓
Text
```

### 2.5 Multilingual Support

Wav2Vec 2.0 itself is an architecture and pretraining approach, not one fixed multilingual recognizer.

Multilingual variants such as XLSR and XLS-R were trained using speech from many languages. The language coverage and recognition quality depend on the particular checkpoint and fine-tuning data.

### 2.6 Advantages and Limitations

| Advantages | Limitations |
|---|---|
| Learns directly from raw audio | Usually requires fine-tuning for useful ASR output |
| Can benefit from unlabelled speech | Pretraining can be computationally expensive |
| Strong speech representations | Checkpoint language coverage varies |
| Useful when labelled data is limited | CTC decoding may need additional language support |

**Key takeaway:** Wav2Vec 2.0 is especially important for learning speech representations from unlabelled audio before supervised ASR fine-tuning.

---

## 3. Whisper

<badge color="success">Major Practical Focus</badge>

![Whisper model illustration](https://images.openai.com/static-rsc-4/46WRKp-NM8NG1SAzCAub0EEvJiLe5bdIImVxfRrhzdZw0pJF-LU3FcIcV--6LDgQLGuR4K8RsWfnYk1euwnmkK5VmKeGVBiz4ST6sW0MM03DLlq4RxHHISqgnlxba_Z4_gxMKh_UDTxWs3ryhwk9T7xXzssq_pkLBR1nAyFtkRc?purpose=inline)

Whisper

An open-source, multilingual, multitask speech recognition model released by OpenAI.

Whisper is one of the most useful models to study because we can download pretrained checkpoints and perform transcription without training an ASR model from scratch.

It was trained on approximately **680,000 hours** of diverse, weakly supervised audio data. Its training includes multilingual transcription, speech translation, language identification and other speech-related tasks.

![OpenAI website icon](https://www.google.com/s2/favicons?domain=https://openai.com&sz=32)

Source: [OpenAI — Introducing Whisper](https://openai.com/index/whisper/)

### 3.1 Architecture

Whisper uses an **Encoder-Decoder Transformer architecture**.

![Whisper encoder-decoder architecture illustration](https://images.openai.com/static-rsc-4/o9H2DDKnsf72TXYz7eaQXwzD481bvf3nntyXhDFBu_onhVN3xHPKCQGveWbXh6oFDsvXplyk8iss1SXm1-XYKn7c3e4Y-k3mJSjyCnWW-Ss3P7j7jqQjWMLY0T7dlfl2g8kA7Jf-CSPLKIjDRDv4xx_Mpp7eo-NxG6Kbgswme_c?purpose=inline)

Source: [Whisper — model overview](https://www.aimodels.fyi/models/replicate/whisper-openai)

It has two main components.

**A. Audio Encoder**

- Receives a Log-Mel spectrogram.
- Processes the acoustic information using Transformer encoder layers.
- Learns contextual representations of the audio.
- Produces encoded audio representations.

**B. Text Decoder**

- Receives special task tokens and previously generated text tokens.
- Uses encoded audio information through cross-attention.
- Predicts the next text token.
- Repeats until the transcription is complete.

### 3.2 Input Representation

Unlike Wav2Vec 2.0, Whisper does not directly feed the raw waveform into its Transformer encoder.

Its input is a **Log-Mel spectrogram**.

```text
Raw Audio
    ↓
Audio Preprocessing
    ↓
STFT
    ↓
Mel Filterbank
    ↓
Logarithmic Transformation
    ↓
Log-Mel Spectrogram
    ↓
Whisper Encoder
```

This connects directly to the MFCC and Log-Mel concepts we studied in Module 34.7.

Whisper uses 80 Mel frequency channels in its standard model configuration. Its audio processing operates on 30-second windows internally.

![OpenAI website icon](https://www.google.com/s2/favicons?domain=https://openai.com&sz=32)

Source: [OpenAI Whisper repository](https://github.com/openai/whisper)

### 3.3 Training Approach

Whisper uses large-scale supervised learning with weakly supervised audio-transcript data.

It is trained on multiple tasks using special tokens.

For example, task tokens can indicate:

- Which language is being spoken.
- Whether the task is transcription or translation.
- Whether the model should produce text or other task-related output.

This allows one model family to handle several speech tasks.

### 3.4 Output Generation

Whisper generates text autoregressively.

**Autoregressive** means that each new token is predicted using the previously generated tokens.

Example:

```text
Audio → Encoder

Decoder:
Token 1 → I
Token 2 → want
Token 3 → to
Token 4 → learn
Token 5 → Python
```

Final output:

```text
I want to learn Python.
```

### 3.5 Multilingual Support

Whisper has multilingual checkpoints that can recognize speech in many languages.

Examples include:

- English
- Tamil
- Hindi
- Telugu
- Malayalam
- Spanish
- French
- Japanese

The accuracy is not equal across all languages, accents and recording conditions. A multilingual model does not guarantee perfect transcription of every language.

Whisper supports two important operations:

**Transcription:**

Tamil audio → Tamil text

**Translation:**

Tamil audio → English text

These are different tasks. We will experiment with both later.

### 3.6 Advantages and Limitations

| Advantages | Limitations |
|---|---|
| Multilingual recognition | Accuracy varies between languages |
| Strong performance without task-specific fine-tuning | Can hallucinate words, especially with silence or unclear audio |
| Robust to many accents and recording conditions | Large checkpoints need substantial memory |
| Supports transcription and translation | Autoregressive decoding can be slower |
| Pretrained checkpoints are publicly available | Domain-specific terminology can still be misrecognized |

### Model Sizes

The original open-source Whisper repository provides several model sizes.

![GitHub icon](https://www.google.com/s2/favicons?domain=https://github.com&sz=32)

Source: [OpenAI Whisper — GitHub](https://github.com/openai/whisper)

| Model | Approx. parameters | Approx. VRAM |
|---|---:|---:|
| Tiny | 39M | 1 GB |
| Base | 74M | 1 GB |
| Small | 244M | 2 GB |
| Medium | 769M | 5 GB |
| Large | 1,550M | 10 GB |
| Turbo | 809M | 6 GB |

These are approximate requirements from the repository, not guaranteed hardware requirements. Actual memory usage varies with inference settings and environment.

For our first practical lab, we'll start with **Base multilingual**. It is a manageable starting point for English and Tamil experiments. We can compare it with Small or Turbo later.

---

## 4. Speech Transformer

The Speech Transformer is an application of the Transformer architecture to speech recognition.

It is important to distinguish this from Whisper:

- **Speech Transformer** refers to a family of Transformer-based ASR architectures.
- **Whisper** is a particular pretrained multilingual Transformer-based ASR model.

### 4.1 Architecture

A typical encoder-decoder Speech Transformer contains:

![Speech Transformer architecture illustration](https://images.openai.com/static-rsc-4/MIfJxnyRx4hnPxsG6K2n0KTH4i6vnLMUsCI6ENQMpAlDh0SYDuTmAHuabg1Zk3aRowmygH3RDdb0Gj5gLK1fU08y29EPC9O_5xctzAVRXrDw2VnXAYAn6Z7SWsoo9KuFmTwU5sPXGW0YZN-zMNtOYnAnwxdbaKGZHqUSFsawzQI?purpose=inline)

Source: [Aman's AI Journal — Speech Processing](https://aman.ai/primers/ai/speech-processing/)

**Encoder:**

- Receives acoustic feature sequences, commonly Log-Mel filterbank features.
- Uses self-attention to learn relationships between different audio frames.
- Generates contextual acoustic representations.

**Decoder:**

- Receives previously generated output tokens.
- Uses masked self-attention.
- Uses cross-attention to access encoder representations.
- Predicts the next text token.

### 4.2 Input Representation

A common input is a sequence of acoustic feature vectors:

$$
X=[x_1,x_2,\ldots,x_T]
$$

Each vector may contain Mel filterbank features.

The model processes the sequence using positional information so that it can learn the order of speech frames.

### 4.3 Training Approach

A typical attention-based Speech Transformer is trained using labelled audio and corresponding text transcripts.

During training:

1. Audio features enter the encoder.
2. The decoder receives the previous reference text tokens.
3. It predicts the next token.
4. The predicted sequence is compared against the target transcript.
5. Cross-entropy loss is used to optimize the model.

This commonly uses **teacher forcing** during training.

### 4.4 Output Generation

The decoder generates text tokens one at a time.

For example:

```text
Audio Features
      ↓
Transformer Encoder
      ↓
Contextual Audio Representations
      ↓
Transformer Decoder
      ↓
I → want → to → learn → AI
      ↓
Recognized Sentence
```

### 4.5 Multilingual Support

A Speech Transformer architecture is not automatically multilingual.

Multilingual capability depends on:

- Training languages.
- Training dataset.
- Tokenizer design.
- Model vocabulary.
- Language-specific fine-tuning.

### 4.6 Advantages and Limitations

| Advantages | Limitations |
|---|---|
| Learns long-range sequence relationships | Attention can be computationally expensive for long audio sequences |
| Parallel processing during encoder training | Autoregressive decoding is sequential |
| End-to-end training | Requires substantial labelled training data |
| Flexible architecture | Performance depends on data and model design |

---

## 5. Comparing the Four Models

Let's bring everything together.

### Architecture Comparison

**DeepSpeech**

```text
Spectral Features
       ↓
Neural Network
       ↓
RNN
       ↓
CTC
       ↓
Characters
```

Historical end-to-end ASR.

**Wav2Vec 2.0**

```text
Raw Waveform
      ↓
CNN
      ↓
Transformer
      ↓
CTC Fine-tuning
      ↓
Text
```

Self-supervised pretraining.

**Whisper**

```text
Audio
  ↓
Log-Mel
  ↓
Transformer Encoder
  ↓
Transformer Decoder
  ↓
Text Tokens
```

Multilingual pretrained ASR.

**Speech Transformer**

```text
Acoustic Features
       ↓
Transformer Encoder
       ↓
Attention Decoder
       ↓
Text
```

Attention-based ASR architecture.

### Detailed Comparison

| Comparison | DeepSpeech | Wav2Vec 2.0 | Whisper | Speech Transformer |
|---|---|---|---|---|
| Input | Spectral features | Raw waveform | Log-Mel | Acoustic features |
| Main architecture | Neural network + RNN | CNN + Transformer | Encoder-decoder Transformer | Encoder-decoder Transformer |
| Training | Supervised + CTC | Self-supervised pretraining + fine-tuning | Large-scale weak supervision | Typically supervised |
| Output | Character probabilities | Usually CTC tokens | Autoregressive text tokens | Autoregressive text tokens |
| Multilingual | Depends on training | Depends on checkpoint | Multilingual checkpoints | Depends on training |
| Main strength | Simple historical CTC approach | Learning from unlabelled audio | Broad multilingual transcription | Attention-based sequence modelling |

My recommendation for our learning roadmap: Understand all four architectures, but spend most of your practical time with Whisper and Wav2Vec 2.0. They introduce two important modern approaches: large-scale pretrained transcription and self-supervised speech representation learning.

---

## 6. Practical Lab — Speech-to-Text Using Whisper 🧪

<badge color="success">Hands-on Implementation</badge>

We'll build a small speech recognition program.

### Our Objectives

1. Load a pretrained Whisper model.
2. Load an audio recording.
3. Perform speech-to-text transcription.
4. Display the recognized text.
5. Compare it with the actual sentence.
6. Experiment with different recordings.

We'll use the original OpenAI Whisper Python package with PyTorch. Its official repository documents this workflow and requires FFmpeg for audio decoding.

![GitHub icon](https://www.google.com/s2/favicons?domain=https://github.com&sz=32)

Source: [OpenAI Whisper — GitHub repository and implementation](https://github.com/openai/whisper/blob/main/README.md)

### 6.1 Install Dependencies

Run this in a Jupyter cell or terminal:

```bash
pip install -U openai-whisper torch jiwer
```

Whisper also requires FFmpeg.

On Windows, you can install it using:

```powershell
winget install Gyan.FFmpeg
```

After installation, restart your terminal and verify:

```powershell
ffmpeg -version
```

If Windows cannot find FFmpeg, check that its `bin` directory is included in your PATH.

### 6.2 Check PyTorch and CUDA

Before loading the model, check whether PyTorch can access your GPU.

```python
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")
```

Select the device:

```python
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Selected Device:", device)
```

### 6.3 Load the Pretrained Whisper Model

```python
import whisper

model = whisper.load_model(
    "base",
    device=device
)

print("Whisper model loaded successfully!")
```

**What happens internally?**

1. Whisper downloads the pretrained model checkpoint if it isn't already cached.
2. The model architecture is initialized.
3. The pretrained weights are loaded.
4. The model is placed on the selected device.

We're using `base` because it is a manageable starting point.

You can experiment with:

```python
# Other model options:
# tiny
# base
# small
# medium
# large
# turbo
```

### 6.4 Load an Audio Recording

Prepare a recording named `speech.wav`.

You can record yourself speaking:

> "Hello, my name is Yuvanesh. I am learning artificial intelligence and machine learning."

Place the file in your project directory.

**Project structure:**

```text
Speech_Recognition/
│
├── speech.wav
│
└── whisper_asr.py
```

### 6.5 Perform Speech-to-Text Transcription

```python
audio_path = "speech.wav"

result = model.transcribe(
    audio_path,
    task="transcribe",
    language="English",
    fp16=(device == "cuda")
)

recognized_text = result["text"]

print("Recognized Text:")
print(recognized_text)
```

**What does `transcribe()` do internally?**

```text
Audio Recording
      ↓
Audio Loading
      ↓
Audio Preprocessing
      ↓
Log-Mel Spectrogram
      ↓
Whisper Encoder
      ↓
Whisper Decoder
      ↓
Text Generation
      ↓
Recognized Sentence
```

The original Whisper implementation handles audio loading, its internal 30-second windowing and autoregressive decoding. You don't need to manually calculate MFCCs or Log-Mel features for this basic inference experiment.

![GitHub icon](https://www.google.com/s2/favicons?domain=https://github.com&sz=32)

Source: [OpenAI Whisper repository](https://github.com/openai/whisper)

### 6.6 Understand the Output

The `result` dictionary contains more than just the transcription.

```python
print(result.keys())
```

You can inspect:

```python
print("Recognized Text:", result["text"])
print("Detected Language:", result["language"])

print("Segments:")

for segment in result["segments"]:
    print(
        segment["start"],
        segment["end"],
        segment["text"]
    )
```

Each segment contains information such as:

- Start time.
- End time.
- Recognized text.
- Additional decoding information.

This is useful when developing transcription and subtitle applications.

### 6.7 Compare Predicted Text with Actual Text

Now comes an important part of machine-learning evaluation.

Suppose the actual sentence was:

```text
Hello, my name is Yuvanesh.
I am learning artificial intelligence and machine learning.
```

But Whisper predicted:

```text
Hello, my name is Yuvanesh.
I am learning artificial intelligence and machine learning.
```

We can compare both sentences using **Word Error Rate (WER)**.

Install:

```bash
pip install jiwer
```

Implementation:

```python
from jiwer import wer

reference = """
Hello, my name is Yuvanesh.
I am learning artificial intelligence and machine learning.
"""

hypothesis = recognized_text

error = wer(reference, hypothesis)

print("Reference:", reference)
print("Predicted:", hypothesis)
print("Word Error Rate:", error)
print("WER Percentage:", error * 100, "%")
```

WER measures the number of word-level errors relative to the reference.

$$
WER=\frac{S+D+I}{N}
$$

Where:

- $S$ = Substitutions.
- $D$ = Deletions.
- $I$ = Insertions.
- $N$ = Total words in the reference.

**For example:**

Reference:

```text
I love machine learning
```

Prediction:

```text
I like machine learning
```

One word is substituted.

$$
WER=\frac{1+0+0}{4}=0.25
$$

<badge color="info">25% WER</badge>

Lower WER generally means better transcription.

Note that WER can exceed 100% when there are enough insertions. Also, punctuation and capitalization can affect raw comparisons, so consistent text normalization is important.

### 6.8 Experiment with Different Recordings

Now let's make the program reusable.

```python
audio_files = [
    "english_speech.wav",
    "tamil_speech.wav",
    "background_noise.wav"
]

for audio_path in audio_files:

    print("\nProcessing:", audio_path)

    result = model.transcribe(
        audio_path,
        task="transcribe",
        fp16=(device == "cuda")
    )

    print("Detected Language:", result["language"])
    print("Recognized Text:", result["text"])
```

For Tamil speech, you can explicitly specify the language:

```python
result = model.transcribe(
    "tamil_speech.wav",
    language="Tamil",
    task="transcribe",
    fp16=(device == "cuda")
)

print(result["text"])
```

Use a multilingual checkpoint such as `base`, not `base.en`, for Tamil.

### 6.9 Experiment with Translation

Whisper can also translate supported non-English speech into English.

For example:

```python
result = model.transcribe(
    "tamil_speech.wav",
    language="Tamil",
    task="translate",
    fp16=(device == "cuda")
)

print("English Translation:")
print(result["text"])
```

This is different from transcription:

| Operation | Input | Output |
|---|---|---|
| Transcription | Tamil speech | Tamil text |
| Translation | Tamil speech | English text |

**Important:** The original Whisper repository notes that `turbo` is not trained for translation tasks. Use a suitable multilingual model such as `base`, `small`, `medium` or `large` for this experiment.

![GitHub icon](https://www.google.com/s2/favicons?domain=https://github.com&sz=32)

Source: [OpenAI Whisper repository](https://github.com/openai/whisper)

---

## 7. Experiments to Perform

Don't stop after one successful transcription. Try these experiments and observe the model's behaviour.

### Experiment Checklist

- [ ] **Clean English recording**

  Record a clear sentence in a quiet environment.

  Objective: Establish baseline transcription quality.

- [ ] **Tamil recording**

  Record 20–30 seconds of natural Tamil speech.

  Objective: Observe multilingual recognition.

- [ ] **Different accents**

  Ask two people to speak the same sentence.

  Objective: Observe accent-related differences.

- [ ] **Background noise**

  Add low-level fan or traffic noise to a recording.

  Objective: Observe noise robustness.

- [ ] **Fast speech**

  Read a paragraph faster than your normal speaking speed.

  Objective: Observe errors caused by speaking speed.

- [ ] **Model comparison**

  Run the same recording through tiny, base and small.

  Objective: Compare speed and transcription quality.

---

## 8. Industry Applications

The models we've studied are used in applications such as:

- **Voice assistants:** Convert spoken commands into text.
- **Meeting transcription:** Generate searchable meeting records.
- **Healthcare:** Assist with medical dictation and documentation.
- **Customer support:** Analyse recorded customer conversations.
- **Accessibility:** Generate captions for videos.
- **Multilingual applications:** Transcribe or translate speech across languages.

For real-world deployment, consider latency, memory usage, language accuracy, privacy, noisy environments and domain-specific vocabulary.

---

## 9. What You Should Remember 🧠

| Model | Most important concept |
|---|---|
| DeepSpeech | Neural ASR with CTC |
| Wav2Vec 2.0 | Self-supervised learning from raw audio |
| Whisper | Multilingual encoder-decoder Transformer |
| Speech Transformer | Attention-based speech-to-text architecture |

And remember:

1. DeepSpeech predicts character probabilities using a neural acoustic model.
2. Wav2Vec 2.0 learns speech representations through self-supervised pretraining.
3. Whisper uses Log-Mel spectrograms and an encoder-decoder Transformer.
4. Speech Transformer applies attention-based sequence modelling to speech.
5. WER measures transcription errors against a reference sentence.
6. Transcription and translation are different tasks.

### Useful Official Resources

- [OpenAI Whisper — GitHub repository and implementation](https://github.com/openai/whisper/blob/main/README.md)
- [Introducing Whisper — OpenAI](https://openai.com/index/whisper/)
- [Whisper — Hugging Face documentation](https://github.com/huggingface/transformers/blob/main/docs/source/en/model_doc/whisper.md)
- [Sequence-to-sequence speech recognition research — Google](https://research.google/pubs/state-of-the-art-speech-recognition-with-sequence-to-sequence-models/)

---

In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Selected Device:", device)

PyTorch Version: 2.7.1+cu128
CUDA Available: True
GPU: NVIDIA GeForce GTX 1650
Selected Device: cuda


In [2]:
import whisper

model = whisper.load_model(
    "base",
    device=device
)

In [5]:
import librosa
from IPython.display import Audio, display

audio, sr = librosa.load(
    "sample-speech-1m.mp3",
    sr= None,
    mono= True
)

print("Sampling Rate: ",sr)
print("Duration:", len(audio) / sr)

display(Audio(audio,rate= sr))

Sampling Rate:  22050
Duration: 60.00539682539682


In [3]:
audio_path = "sample-speech-1m.mp3"

result = model.transcribe(
    audio_path,
    task="transcribe",
    language="English",
    fp16=(device == "cuda")
)

recognized_text = result["text"]

print("Recognized Text:")
print(recognized_text)

Recognized Text:
 Welcome to samplolid.com, a free online resource for downloading sample files in a wide variety of digital formats, whether you are a software developer testing file upload functionality, a quality assurance engineer validating media players, a student learning about digital formats, or simply someone who needs a quick test file, samplolid provides ready-to-use files that you can download instantly, completely free of charge. In this recording, we will walk you through every aspect of the samplolid platform, exploring the formats we offer, the technical details behind each one, and the many ways these test files can be used in your projects and workflows. Let us begin with image formats. Images are perhaps the most fundamental type of digital media, and samplolid offers test files in five major image formats, JPEG, PNG, SVG, GIF, and web. Image format has its own strengths, trade-offs, and ideal use cases, and understanding these differences as essential.


In [6]:
result.keys()

dict_keys(['text', 'segments', 'language'])

In [7]:
print("Recognized Text:", result["text"])

print("Detected Language:", result["language"])

print("Segments:")

for segment in result["segments"]:

    print(
        segment["start"],
        segment["end"],
        segment["text"]
    )

Recognized Text:  Welcome to samplolid.com, a free online resource for downloading sample files in a wide variety of digital formats, whether you are a software developer testing file upload functionality, a quality assurance engineer validating media players, a student learning about digital formats, or simply someone who needs a quick test file, samplolid provides ready-to-use files that you can download instantly, completely free of charge. In this recording, we will walk you through every aspect of the samplolid platform, exploring the formats we offer, the technical details behind each one, and the many ways these test files can be used in your projects and workflows. Let us begin with image formats. Images are perhaps the most fundamental type of digital media, and samplolid offers test files in five major image formats, JPEG, PNG, SVG, GIF, and web. Image format has its own strengths, trade-offs, and ideal use cases, and understanding these differences as essential.
Detected Lan

In [12]:
# Word Error Rate (WER)

from jiwer import wer

reference = """Welcome to samplolid.com, a free online resource for downloading sample files in a wide variety of digital formats,
whether you are a software developer testing file upload functionality, a quality assurance engineer validating media players, 
a student learning about digital formats, or simply someone who needs a quick test file, samplolid provides ready-to-use files that you can download instantly,
completely free of charge. In this recording, we will walk you through every aspect of the samplolid platform, exploring the formats we offer, 
the technical details behind each one, and the many ways these test files can be used in your projects and workflows. Let us begin with image formats. 
Images are perhaps the most fundamental type of digital media, and samplolid offers test files in five major image formats, JPEG, PNG, SVG, GIF, and web. 
Image format has its own strengths, trade-offs, and ideal use cases, 
and understanding these differences are essential."""

hypothesis = recognized_text

error = wer(reference, hypothesis)

print("Reference:", reference)
print("\nPredicted:", hypothesis)
print("\nWord Error Rate:", error)
print("\nWER Percentage:", error * 100, "%")

Reference: Welcome to samplolid.com, a free online resource for downloading sample files in a wide variety of digital formats,
whether you are a software developer testing file upload functionality, a quality assurance engineer validating media players, 
a student learning about digital formats, or simply someone who needs a quick test file, samplolid provides ready-to-use files that you can download instantly,
completely free of charge. In this recording, we will walk you through every aspect of the samplolid platform, exploring the formats we offer, 
the technical details behind each one, and the many ways these test files can be used in your projects and workflows. Let us begin with image formats. 
Images are perhaps the most fundamental type of digital media, and samplolid offers test files in five major image formats, JPEG, PNG, SVG, GIF, and web. 
Image format has its own strengths, trade-offs, and ideal use cases, 
and understanding these differences are essential.

Predicted:  

In [17]:
# Experiment with different recordings

audio_files = [
    "example.mp3",
    "sample-speech-1m.mp3",
    "tamil_audio.mp3"
]

for audio_path in audio_files:

    print("\nProcessing:", audio_path)

    result = model.transcribe(
        audio_path,
        task="transcribe",
        fp16=(device == "cuda")
    )

    print("Detected Language:", result["language"])
    print("Recognized Text:", result["text"])


Processing: example.mp3
Detected Language: en
Recognized Text:  bioته

Processing: sample-speech-1m.mp3
Detected Language: en
Recognized Text:  Welcome to samplolid.com, a free online resource for downloading sample files in a wide variety of digital formats, whether you are a software developer testing file upload functionality, a quality assurance engineer validating media players, a student learning about digital formats, or simply someone who needs a quick test file, samplolid provides ready-to-use files that you can download instantly, completely free of charge. In this recording, we will walk you through every aspect of the samplolid platform, exploring the formats we offer, the technical details behind each one, and the many ways these test files can be used in your projects and workflows. Let us begin with image formats. Images are perhaps the most fundamental type of digital media, and samplolid offers test files in five major image formats, JPEG, PNG, SVG, GIF, and web. Imag

In [18]:
result = model.transcribe(
    "tamil_audio.mp3",
    language="Tamil",
    task="translate",
    fp16=(device == "cuda")
)

print("English Translation:")
print(result["text"])

English Translation:
 the grilled isn't the state of the
